# Kaggriculture — coordinated market-aware farming

Standalone Python agent. Uses crop profitability forecasts, coordinated workers, conservative seed purchasing and explicit final-day liquidation. Local results are not a leaderboard ranking. Run all cells, save a notebook version, then submit `submission.tar.gz` to Kaggriculture.

In [ ]:
from pathlib import Path

AGENT_SOURCE = '"""Kaggriculture: coordinated, market-aware farm policy. Standard library only.\n\nv2 of main.py. Changes are confined to market execution: the original sold the\nentire shed every turn at whatever the spot price was, which walked MELON from\n272 down to 60 over a season (its glut term is quadratic, and T is only 300).\nSelling is now metered against the marginal price curve, while still clearing\nstock before the season ends.\n"""\nimport math\n\n# seed cost, first harvest, peak harvest, peak units, last harvest\nCROPS = {\'WHEAT\':(10,2,4,4,4), \'CARROT\':(20,2,3,3,3),\n         \'MELON\':(80,10,10,6,12), \'TOMATO\':(50,8,11,4,11),\n         \'STRAWBERRY\':(100,10,16,4,16)}\nPARAMS = {\n \'WHEAT\':(25,400,\'sqrt\',.8,\'log\',.2), \'CARROT\':(35,450,\'hinge\',1.,\'sqrt\',.7),\n \'TOMATO\':(60,200,\'hinge\',.4,\'sqrt\',.6), \'STRAWBERRY\':(120,100,\'sqrt\',.7,\'linear\',1.6),\n \'MELON\':(250,300,\'log\',.2,\'sq\',3.6), \'EGG\':(50,332,\'hinge\',.4,\'log\',.2),\n \'MILK\':(160,122,\'sqrt\',.6,\'linear\',1.6), \'WOOL\':(200,105,\'log\',.2,\'sq\',3.2),\n \'FERTILIZER\':(100,200,\'linear\',.4,\'linear\',.4)}\n# Livestock. GOOSE is omitted: ~27 coins per worker-action against MELON\'s ~118\n# and COW\'s ~160. CARE banks a +1 bonus that the next fed production tick\n# consumes, doubling output; an animal unfed two days escapes and is lost.\nANIMALS = {\n    \'COW\':   {\'cost\':400,\'pen\':\'PASTURE\',\'first\':8,\'iv\':2,\'held\':6,\'prod\':\'MILK\'},\n    \'SHEEP\': {\'cost\':500,\'pen\':\'PASTURE\',\'first\':6,\'iv\':3,\'held\':6,\'prod\':\'WOOL\'},\n    # GOOSE was excluded on coins-per-worker-action, which ignored how often it\n    # pays: it yields EVERY day from day 4, where a cow yields every second day\n    # from day 8. That makes it the only livestock that returns cash during the\n    # opening, when the farm sits on ~150 coins and cannot afford anything.\n    \'GOOSE\': {\'cost\':300,\'pen\':\'COOP\',\'first\':4,\'iv\':1,\'held\':4,\'prod\':\'EGG\'},\n}\nSHOPS = {\'BAKERY\':[\'EGG\',\'WHEAT\'], \'PIZZA_SHOP\':[\'MILK\',\'TOMATO\',\'WHEAT\'],\n \'BRUNCH_SPOT\':[\'EGG\',\'WHEAT\',\'STRAWBERRY\'], \'YARN_STORE\':[\'WOOL\',\'WOOL\'],\n \'ICE_CREAM_SHOP\':[\'STRAWBERRY\',\'MILK\',\'WHEAT\'], \'PET_CAFE\':[\'CARROT\',\'CARROT\'],\n \'SMOOTHIE_SHOP\':[\'STRAWBERRY\',\'MILK\'], \'FARMERS_MARKET\':[\'WHEAT\',\'CARROT\',\'TOMATO\',\'STRAWBERRY\']}\n# Tunable vector (see tune.py). Named TUNE, not PARAMS: PARAMS above is this\n# agent\'s market price table and must not be shadowed.\nTUNE = {\n    \'workers_hi\': 10.61, \'workers_mid\': 4.676, \'workers_lo\': 4.707,\n    \'active_hi\': 5.578, \'active_lo\': 4.0,\n    \'max_land\': 2.487, \'land_cash\': 1405, \'land_last_day\': 20.31, \'land_fill\': 0.3983,\n    \'sell_floor\': 0.6263, \'sell_spread\': 2.297, \'sell_clear\': 0.2318,\n    \'carry_cap\': 12.52, \'travel_cost\': 1.204, \'seed_cap\': 4.261,\n    \'water_urgent\': 251, \'water_base\': 69.73, \'opp_weight\': 1.0,\n    \'herd\': 7.245, \'barn_r\': 2.882, \'animal_last_day\': 12.57, \'animal_cash\': 2754,\n    \'feed_batch\': 6.401, \'animal_harvest\': 5.948, \'feed_days\': 3.0, \'sheep_share\': 0.1251,\n    \'labor_weight\': 0.0,   # 0 = per-tile-day scoring, 1 = per-action (1.0 measured far worse)\n    \'feed_horizon\': 19.73, \'feed_safety\': 1.0,\n    # Tier-3 task priorities. These were hand-set guesses; the learned linear\n    # model had already found PLACE/PICKUP were ~10x too high, so they are\n    # parameters now and searched like everything else.\n    \'v_pen\': 500.0, \'v_plant\': 2.0, \'v_aharv\': 2.0, \'v_feed_u\': 30.0,\n    \'v_feed\': 14.0, \'v_care\': 5.0, \'v_place\': 40000.0, \'v_water_h\': 250.0,\n    \'v_harv\': 300.0, \'v_harv_m\': 2.0, \'v_pick_w\': 6000.0, \'v_pick_a\': 20000.0,\n    \'goose_share\': 0.0,   # measured worse: a daily feed+care+collect for a 50-coin egg\n    \'feed_buffer\': 2.001, \'feed_max\': 24.0,\n    \'optimal_assign\': 1,   # 0 = greedy crew assignment, 1 = Hungarian\n    \'v_fert\': 4.646,   # weight on collecting fertilizer from owned animals\n    \'plant_radius\': 8.043,   # only sow within this distance of the shed\n}\nSPACE = [\n    (\'workers_hi\', 4.0, 13.0), (\'workers_mid\', 3.0, 11.0), (\'workers_lo\', 2.0, 9.0),\n    (\'active_hi\', 2.0, 40.0), (\'active_lo\', 1.0, 20.0),\n    (\'max_land\', 2.0, 4.0), (\'land_cash\', 100.0, 4000.0),\n    (\'land_last_day\', 6.0, 24.0), (\'land_fill\', 0.2, 1.2),\n    (\'sell_floor\', 0.30, 1.10), (\'sell_spread\', 1.0, 10.0), (\'sell_clear\', 0.0, 4.0),\n    (\'carry_cap\', 10.0, 95.0), (\'travel_cost\', 0.5, 5.0), (\'seed_cap\', 3.0, 20.0),\n    (\'water_urgent\', 100.0, 900.0), (\'water_base\', 20.0, 400.0),\n    (\'opp_weight\', 0.85, 1.15),\n    (\'herd\', 0.0, 16.0), (\'barn_r\', 2.0, 5.0), (\'animal_last_day\', 4.0, 16.0),\n    (\'animal_cash\', 200.0, 3000.0), (\'feed_batch\', 2.0, 14.0),\n    (\'animal_harvest\', 1.0, 6.0), (\'labor_weight\', 0.0, 0.5),\n    (\'feed_days\', 0.0, 8.0), (\'sheep_share\', 0.0, 0.6),\n    (\'plant_radius\', 3.0, 10.0),\n]\n# Worker-actions one tile of each crop consumes over its life: plant, the waters\n# that actually add yield, the survival waters between them, and harvest(s).\n# Labour is the binding constraint -- land is 4000 a quadrant while the 13th\n# hand costs ~7000 a season -- so crops are ranked per action, not per tile-day.\nLABOR = {\'WHEAT\':6.0, \'CARROT\':5.0, \'MELON\':12.0, \'TOMATO\':11.0, \'STRAWBERRY\':14.0}\n# ---------------------------------------------------------------- learned policy\n# Actions are chosen autoregressively: units are assigned one at a time and each\n# candidate is scored with the assignments already made this turn as input, so\n# the policy can spread the crew out or pile onto urgent work.\n#\n# The score is the hand-built heuristic multiplied by exp(W . phi). W is all\n# zeros by default, which reproduces the heuristic exactly -- training can only\n# move away from a known-good baseline, never start from noise.\n#\n# phi, in order (both plant and animal work share one feature space, so the\n# model trades a cow\'s feeding against a melon\'s watering on the same scale):\n#   0 water plant      1 harvest plant    2 plant seed     3 feed animal\n#   4 care animal      5 harvest animal   6 fetch/place    7 build pen\n#   8 distance/10      9 urgent          10 day progress  11 hour progress\n#  12 crowding        13 same-op already 14 yield/6       15 bias\nNW = 26\n# Learned by train_policy.py: 151-9 against the zero-weight heuristic\n# in training, 54-6 (p=1e-10) on held-out seeds. The model prefers\n# high-yield tiles and prompt logistics, spreads the crew out\n# (negative crowding), and works harder later in the day.\nW = [+0.289229, +0.242076, +0.662843, +0.545308, -0.366402, -0.148995, -2.291418, +0.664476, -0.418600, -0.171905, +0.808082, -0.007582, -0.611780, +0.103980, +0.695016, -0.502503, +0.546291, +0.243043, -0.261623, +0.078200, -0.648019, -0.224436, -0.017673, +0.156826, -0.218265, -0.037267]\n\n\n\n\n\n\ndef shape(kind, x, t):\n    x=max(0,x)\n    if kind==\'sqrt\': return math.sqrt(x)\n    if kind==\'sq\': return x*x\n    if kind==\'log\': return math.log1p(x)\n    if kind==\'log10\': return math.log10(1+x)\n    if kind==\'hinge\': return x/t+8*max(0,x/t-1)**2\n    return x\n\n\ndef price(crop, inventory, overrides=None):\n    base,t,lo,lt,hi,ht=PARAMS[crop]\n    p=(overrides or {}).get(crop,{})\n    base=p.get(\'base\',base); t=p.get(\'T\',t); anchor=p.get(\'I0\',10000)\n    below=inventory<anchor\n    f=p.get(\'below_func\' if below else \'above_func\',lo if below else hi)\n    target=p.get(\'below_target\' if below else \'above_target\',lt if below else ht)\n    return max(1,round(base+(1 if below else -1)*base*target*shape(f,abs(inventory-anchor),t)/shape(f,t,t)))\n\n\ndef task_features(op, tile, day, hour, tpd, final_day):\n    """The part of phi that depends only on the task, so it is computed once per\n    task per turn instead of once per (unit, task) pair -- ~9x fewer calls."""\n    return features(op, 0, tile, day, hour, tpd, final_day, 0, 0)\n\n\n\ndef hungarian(cost):\n    """Minimum-cost perfect assignment of rows to columns (Jonker-Volgenant\n    shortest augmenting path). cost is a rectangular list of lists with\n    len(rows) <= len(cols); returns a list giving each row\'s column.\n\n    The crew is assigned greedily today: worker 1 takes the best task, worker 2\n    the best of what is left, and so on. That is a 1/2-approximation -- an early\n    worker can take a job a later worker was far better placed for, and nothing\n    can undo it. With at most ~10 workers and a few dozen candidate tasks the\n    exact solve costs microseconds, so there is no reason to approximate.\n    """\n    n = len(cost)\n    if n == 0:\n        return []\n    m = len(cost[0])\n    INF = float(\'inf\')\n    u = [0.0]*(n+1)\n    v = [0.0]*(m+1)\n    p = [0]*(m+1)\n    way = [0]*(m+1)\n    for i in range(1, n+1):\n        p[0] = i\n        j0 = 0\n        minv = [INF]*(m+1)\n        used = [False]*(m+1)\n        while True:\n            used[j0] = True\n            i0 = p[j0]\n            delta = INF\n            j1 = 0\n            for j in range(1, m+1):\n                if not used[j]:\n                    cur = cost[i0-1][j-1] - u[i0] - v[j]\n                    if cur < minv[j]:\n                        minv[j] = cur\n                        way[j] = j0\n                    if minv[j] < delta:\n                        delta = minv[j]\n                        j1 = j\n            if delta == INF:\n                break\n            for j in range(m+1):\n                if used[j]:\n                    u[p[j]] += delta\n                    v[j] -= delta\n                else:\n                    minv[j] -= delta\n            j0 = j1\n            if p[j0] == 0:\n                break\n        while j0:\n            j1 = way[j0]\n            p[j0] = p[j1]\n            j0 = j1\n    out = [-1]*n\n    for j in range(1, m+1):\n        if 1 <= p[j] <= n:\n            out[p[j]-1] = j-1\n    return out\n\n\ndef features(op, dist, tile, day, hour, tpd, final_day, crowd, same_op):\n    """phi for one candidate action, given what this turn has already assigned."""\n    p = [0.0]*NW\n    is_animal = isinstance(tile, dict) and \'animal\' in tile\n    if op == \'WATER\': p[0] = 1.0\n    elif op == \'HARVEST\': p[5 if is_animal else 1] = 1.0\n    elif op == \'PLANT\': p[2] = 1.0\n    elif op == \'FEED\': p[3] = 1.0\n    elif op == \'CARE\': p[4] = 1.0\n    elif op in (\'PLACE\', \'PICKUP\'): p[6] = 1.0\n    elif op == \'BUILD_PASTURE\': p[7] = 1.0\n    p[8] = dist / 10.0\n    if isinstance(tile, dict):\n        if tile.get(\'consecutive_unwatered\', 0) >= 1 or tile.get(\'consecutive_unfed\', 0) >= 1:\n            p[9] = 1.0\n        p[14] = min(1.0, tile.get(\'yield_units\', 0) / 6.0)\n    p[10] = min(1.0, day / max(1.0, float(final_day)))\n    p[11] = hour / float(tpd)\n    p[12] = min(1.0, crowd / 4.0)\n    p[13] = min(1.0, same_op / 4.0)\n    p[15] = 1.0\n    # Rule-derived facts about the tile\'s situation. These are consequences of\n    # the engine\'s rules, like "is this piece attacked" -- they describe the\n    # position, they do not rank it. The correction network decides what they\n    # are worth. Appended after index 15 so the weights learned over the\n    # original sixteen features keep their meaning.\n    crop = tile.get(\'crop\') if isinstance(tile, dict) else None\n    if crop in CROPS:\n        cost, first, peak, units, last = CROPS[crop]\n        age = day - tile.get(\'planted_day\', day)\n        yu = tile.get(\'yield_units\', 0)\n        p[16] = min(1.0, max(0, units - yu) / 6.0)          # yield headroom\n        p[17] = min(1.0, max(0, first - age) / 12.0)        # turns to maturity\n        lo = 2 if crop in (\'WHEAT\', \'CARROT\') else 6\n        p[18] = 1.0 if lo <= age <= last else 0.0           # watering pays today\n        p[19] = 1.0 if crop in (\'TOMATO\', \'STRAWBERRY\') else 0.0\n        mls = tile.get(\'max_lifespan_step\', -1)\n        if mls >= 0:\n            p[20] = min(1.0, max(0, mls - (day * tpd + hour)) / (4.0 * tpd))\n        p[21] = min(1.0, cost / 100.0)\n    if is_animal:\n        p[22] = min(1.0, tile.get(\'consecutive_unfed\', 0) / 2.0)\n        p[23] = 1.0 if tile.get(\'cared_today\') else 0.0\n    p[24] = min(1.0, day / max(1.0, float(final_day)))\n    p[25] = 1.0 if day >= final_day else 0.0\n    return p\n\n\ndef distance(a,b): return abs(a[0]-b[0])+abs(a[1]-b[1])\n\n\ndef move(a,b):\n    if a[0]<b[0]: return [\'EAST\']\n    if a[0]>b[0]: return [\'WEST\']\n    if a[1]<b[1]: return [\'SOUTH\']\n    if a[1]>b[1]: return [\'NORTH\']\n    return [\'PASS\']\n\n\ndef agent(obs, configuration=None):\n    cfg=configuration or {}\n    if not obs.get(\'farms\'): return {\'farmer\':[\'PASS\'],\'hands\':[],\'market\':[]}\n    farm=obs[\'farms\'][obs[\'player\']]; private=obs[\'private\']\n    tiles=farm[\'tiles\']; n=len(tiles); half=n//2\n    shed_positions=[(half-1,half-1),(half,half-1),(half-1,half),(half,half)]\n    positions=[farm[\'farmer\']]+farm.get(\'hands\',[])\n    inventories=private[\'inventories\']; seeds=dict(private.get(\'seeds\',{}))\n    shed=private.get(\'shed\',{}); day=obs[\'day\']; hour=obs[\'hour\']\n    tpd=cfg.get(\'turnsPerDay\',24); step=obs.get(\'step\',day*tpd+hour)\n    final_step=cfg.get(\'episodeSteps\',720)-2\n    days_left=max(0,(final_step-step)//tpd)\n    final_day=final_step//tpd; final_hour=final_step%tpd\n    remaining=(final_hour-hour+1) if day==final_day else tpd-hour\n    market_inv=obs[\'market\'].get(\'inventory\',{}); prices=obs[\'market\'][\'prices\']\n    overrides=cfg.get(\'marketParams\',{})\n    demand={c:1.0 for c in PARAMS}; demand[\'FERTILIZER\']=0\n    for shop in obs.get(\'town\',{}).get(\'unlocked_shops\',[]):\n        for c in SHOPS.get(shop,[]): demand[c]+=tpd/cfg.get(\'townShopSellInterval\',4)\n    n_animals=n_pens=0\n    empty_pens={\'PASTURE\':0,\'COOP\':0}\n    for _row in tiles:\n        for _t in _row:\n            if isinstance(_t,dict):\n                if \'animal\' in _t: n_animals+=1\n                elif _t.get(\'kind\') in (\'COOP\',\'PASTURE\'):\n                    n_pens+=1; empty_pens[_t[\'kind\']]+=1\n    owned={\'COW\':0,\'SHEEP\':0,\'GOOSE\':0}\n    for _r in tiles:\n        for _t in _r:\n            if isinstance(_t,dict) and _t.get(\'animal\') in owned: owned[_t[\'animal\']]+=1\n    _tot=owned[\'COW\']+owned[\'SHEEP\']+owned[\'GOOSE\']\n    if TUNE[\'goose_share\']*(_tot+1)>owned[\'GOOSE\']: want_animal=\'GOOSE\'\n    elif TUNE[\'sheep_share\']*(_tot+1)>owned[\'SHEEP\']: want_animal=\'SHEEP\'\n    else: want_animal=\'COW\'\n    want_kind=ANIMALS[want_animal][\'pen\']\n    want_pens=0\n    if day<=TUNE[\'animal_last_day\'] and farm[\'money\']>TUNE[\'animal_cash\']:\n        want_pens=max(0,int(TUNE[\'herd\'])-n_animals-n_pens)\n    commitments={c:0.0 for c in CROPS}\n    own_count={c:0 for c in CROPS}\n    for fi,f in enumerate(obs[\'farms\']):\n        for row in f[\'tiles\']:\n            for tile in row:\n                if isinstance(tile,dict) and tile.get(\'kind\')==\'PLANT\':\n                    c=tile[\'crop\']; age=day-tile[\'planted_day\']; data=CROPS[c]\n                    if age<=data[4]:\n                        commitments[c]+=max(tile.get(\'yield_units\',0),data[3]*(TUNE[\'opp_weight\'] if fi!=obs[\'player\'] else 1))\n                        if fi==obs[\'player\']: own_count[c]+=1\n    def crop_score(c):\n        cost,first,peak,units,last=CROPS[c]\n        horizon=min(peak,days_left)\n        if days_left<first+1: return -1000\n        if c in (\'TOMATO\',\'STRAWBERRY\'):\n            units=min(4,1+(horizon-first)//(1 if c==\'TOMATO\' else 2))\n        elif c==\'WHEAT\': units=min(4,max(1,horizon-1))\n        elif c==\'CARROT\': units=min(3,max(1,horizon))\n        forecast=market_inv.get(c,10000)+commitments[c]-demand[c]*horizon*.8\n        estimated=(price(c,forecast,overrides)+price(c,forecast+units,overrides))/2\n        profit=estimated*units-cost\n        # Blend tile-days and worker-actions. At labor_weight=1 this is pure\n        # coins-per-action, which ranks MELON (~118) far above STRAWBERRY (~27).\n        w=TUNE[\'labor_weight\']\n        return profit/((horizon+1)**(1-w)*LABOR[c]**w)\n    crop_values={c:crop_score(c) for c in CROPS}\n    best_crop=max(CROPS,key=lambda c:crop_values[c])\n    best_value=crop_values[best_crop]\n    active=sum(own_count.values())\n    unlocked=len(farm.get(\'unlocked_quadrants\',[\'NW\']))\n    money=farm[\'money\']; orders=[]; keep=[]; expected_drop={}\n    feed_hold=n_animals*max(1,prices.get(\'WHEAT\',25))*TUNE[\'feed_days\']\n    # Cash and stock accounting includes same-turn deliveries: actions precede market.\n    reserved=set(); tasks=[]; op_count={}\n    for y,row in enumerate(tiles):\n        for x,tile in enumerate(row):\n            target=(x,y)\n            if tile==\'LOCKED\': continue\n            if tile is None or (isinstance(tile,dict) and tile.get(\'kind\')==\'WEED\'):\n                if (tile is None and want_pens>0 and remaining>=3\n                        and distance(target,shed_positions[0])<=TUNE[\'barn_r\']):\n                    want_pens-=1\n                    tasks.append((target,\'BUILD_\'+want_kind,TUNE[\'v_pen\'],None))\n                elif (best_value>2 and remaining>=3\n                        and distance(target,shed_positions[0])<=TUNE[\'plant_radius\']):\n                    tasks.append((target,\'PLANT\',max(4,best_value*TUNE[\'v_plant\']),best_crop))\n                continue\n            if \'animal\' in tile:\n                ad=ANIMALS.get(tile[\'animal\'])\n                if ad is None: continue\n                pprice=prices.get(ad[\'prod\'],ad[\'cost\'])\n                ayu=tile.get(\'yield_units\',0)\n                # Every animal produces one fertilizer a day, unconditionally --\n                # the engine sets fertilizer_available on every animal at every\n                # end of day, fed or not. It is free output from livestock the\n                # farm already owns and was being left on the ground.\n                if tile.get(\'fertilizer_available\') and day!=final_day:\n                    tasks.append((target,\'COLLECT_FERTILIZER\',\n                                  prices.get(\'FERTILIZER\',100)*TUNE[\'v_fert\'],None))\n                if ayu>=TUNE[\'animal_harvest\'] or ayu>=ad[\'held\'] or (day==final_day and ayu>0):\n                    tasks.append((target,\'HARVEST\',ayu*pprice*TUNE[\'v_aharv\'],None))\n                elif not tile.get(\'fed_today\') and day!=final_day:\n                    # Fed every day on purpose. Feeding only every other day\n                    # (enough to survive) was measured at 112 milk against 168,\n                    # because the CARE bonus is consumed only on a fed\n                    # production tick -- the lost milk far outweighs the wheat.\n                    urgent=tile.get(\'consecutive_unfed\',0)>=1\n                    tasks.append((target,\'FEED\',pprice*(TUNE[\'v_feed_u\'] if urgent else TUNE[\'v_feed\']),\'WHEAT\'))\n                elif not tile.get(\'cared_today\') and day!=final_day:\n                    tasks.append((target,\'CARE\',pprice*TUNE[\'v_care\'],None))\n                continue\n            if tile.get(\'kind\') in (\'COOP\',\'PASTURE\'):\n                # One task per species that fits this pen. COW and SHEEP share\n                # PASTURE, so offering only the first leaves a unit carrying the\n                # other unable to place it: it loops on PICKUP and end-of-day\n                # auto-drop returns the animal to the shed, forever.\n                for a in ANIMALS:\n                    if ANIMALS[a][\'pen\']==tile.get(\'kind\'):\n                        tasks.append((target,\'PLACE\',TUNE[\'v_place\'],a))\n                continue\n            if tile.get(\'kind\')!=\'PLANT\': continue\n            c=tile[\'crop\']; cost,first,peak,units,last=CROPS[c]; age=day-tile[\'planted_day\']\n            yield_units=tile.get(\'yield_units\',0)\n            mature=age>=first and yield_units>0\n            end=(day==final_day)\n            harvest=mature and (age>=peak or end or tile.get(\'max_lifespan_step\',-1)>=0 and step>=tile[\'max_lifespan_step\']-tpd)\n            if harvest:\n                # Water before harvest if it adds an immediate one-time yield.\n                window=2 if c in (\'WHEAT\',\'CARROT\') else 6\n                cap=6 if c in (\'WHEAT\',\'MELON\') else 4\n                if c not in (\'TOMATO\',\'STRAWBERRY\') and not tile[\'watered_today\'] and window<=age<=last and yield_units<cap:\n                    tasks.append((target,\'WATER\',max(TUNE[\'v_water_h\'],prices.get(c,25)*yield_units),None))\n                else: tasks.append((target,\'HARVEST\',max(TUNE[\'v_harv\'],prices.get(c,25)*yield_units*TUNE[\'v_harv_m\']),None))\n            elif not tile.get(\'watered_today\') and not end:\n                urgency=TUNE[\'water_urgent\'] if tile.get(\'consecutive_unwatered\',0)>=1 else TUNE[\'water_base\']\n                tasks.append((target,\'WATER\',urgency+max(0,prices.get(c,25)*.2),None))\n    hungry=sum(1 for _r in tiles for _t in _r\n               if isinstance(_t,dict) and \'animal\' in _t and not _t.get(\'fed_today\'))\n    carried_wheat=sum(i.get(\'WHEAT\',0) for i in inventories)\n    # Only a unit with wheat IN HAND may feed, so what matters is how many\n    # units carry some, not how much wheat exists. Comparing hungry animals to\n    # the total let one unit hold ten wheat and satisfy the test, leaving the\n    # other eight units permanently unable to feed: in a real game 8 animals\n    # stayed hungry all day beside 28 wheat in the shed, and four cows starved.\n    carriers=sum(1 for i in inventories if i.get(\'WHEAT\',0)>0)\n    # An animal that missed yesterday dies tonight if it misses again, and only\n    # a unit with wheat in hand may feed. Spreading wheat over more units costs\n    # extra shed trips, so it is done only for animals actually at risk, not\n    # for every animal that simply has not eaten yet today.\n    at_risk=sum(1 for _r in tiles for _t in _r\n                if isinstance(_t,dict) and \'animal\' in _t\n                and not _t.get(\'fed_today\') and _t.get(\'consecutive_unfed\',0)>=1)\n    if ((hungry>carried_wheat or at_risk>carriers)\n            and shed.get(\'WHEAT\',0)>0 and day!=final_day):\n        take=int(min(shed[\'WHEAT\'],max(1,TUNE[\'feed_batch\'])))\n        for st in shed_positions:\n            tasks.append((st,\'PICKUP\',TUNE[\'v_pick_w\'],(\'WHEAT\',take)))\n    carried_animals=sum(i.get(a,0) for i in inventories for a in ANIMALS)\n    if carried_animals==0 and n_pens>0:\n        for a in ANIMALS:\n            if shed.get(a,0)>0:\n                for st in shed_positions:\n                    tasks.append((st,\'PICKUP\',TUNE[\'v_pick_a\'],(a,shed[a])))\n    # W . phi, precomputed per task. phi differs per unit only at indices 8\n    # (distance), 12 (crowding) and 13 (same op already chosen), so the dot\n    # product is reused and the inner loop does three multiplies instead of 26.\n    base_dot=[]\n    for (t,op,_v,_c) in tasks:\n        ph=task_features(op,tiles[t[1]][t[0]],day,hour,tpd,final_day)\n        base_dot.append(sum(W[k]*ph[k] for k in range(NW)))\n\n    # Optimal assignment of the whole crew at once. Scores are built for every\n    # (worker, task) pair that is legal for that worker -- a worker with no\n    # wheat cannot FEED, one with no seed cannot PLANT -- then Hungarian picks\n    # the best set of pairs jointly instead of first-come-first-served.\n    assign={}\n    if TUNE[\'optimal_assign\']:\n        free=[i for i,pos in enumerate(positions)\n              if not (sum((inventories[i] if i<len(inventories) else {}).values())\n                      and (day==final_day\n                           and remaining<=min(distance(pos,q) for q in shed_positions)+2\n                           or sum((inventories[i] if i<len(inventories) else {}).values())\n                           >=TUNE[\'carry_cap\']))]\n        BIG=1e9\n        rows=[]\n        for i in free:\n            pos=positions[i]\n            inv=inventories[i] if i<len(inventories) else {}\n            row=[]\n            for ti,(target,op,value,c) in enumerate(tasks):\n                dist=distance(pos,target)\n                needed=2+(1 if tiles[target[1]][target[0]] is not None else 0) if op==\'PLANT\' else 1\n                ok=True\n                if op==\'PLANT\' and seeds.get(c,0)<=0: ok=False\n                if op==\'FEED\' and inv.get(\'WHEAT\',0)<=0: ok=False\n                if op==\'PLACE\' and inv.get(c,0)<=0: ok=False\n                if dist+needed>remaining: ok=False\n                if not ok: row.append(BIG); continue\n                z=base_dot[ti]+W[8]*(dist/10.0)\n                if z>3.0: z=3.0\n                elif z<-3.0: z=-3.0\n                row.append(-value/(dist+TUNE[\'travel_cost\'])*math.exp(z))\n            rows.append(row)\n        if rows and tasks:\n            # Pad so every worker has a column to take (doing nothing).\n            width=len(tasks)+len(rows)\n            for k,row in enumerate(rows):\n                row.extend([0.0 if j==k else BIG for j in range(len(rows))])\n            sol=hungarian([r[:width] for r in rows])\n            # The joint solve has no notion of a shared stock, so it will\n            # happily send three workers to plant one seed -- and the engine\n            # drops ALL of a crop\'s PLANTs when more are ordered than held.\n            # Keep the best-scoring ones and let the rest fall back to greedy.\n            picked=sorted(((rows[k][sol[k]],k) for k in range(len(free))\n                           if 0<=(sol[k] if k<len(sol) else -1)<len(tasks)\n                           and rows[k][sol[k]]<BIG/2))\n            budget=dict(seeds)\n            for _sc,k in picked:\n                j=sol[k]; target,op,value,c=tasks[j]\n                if op==\'PLANT\':\n                    if budget.get(c,0)<=0: continue\n                    budget[c]-=1\n                assign[free[k]]=j\n    actions=[]\n    for idx,pos in enumerate(positions):\n        inv=inventories[idx] if idx<len(inventories) else {}\n        closest=min(shed_positions,key=lambda p:distance(pos,p)); home_dist=distance(pos,closest)\n        carrying=sum(inv.values())\n        if carrying and (day==final_day and remaining<=home_dist+2 or carrying>=TUNE[\'carry_cap\']):\n            action=[\'DROP\'] if home_dist==0 else move(pos,closest)\n            if home_dist==0:\n                for c,q in inv.items(): expected_drop[c]=expected_drop.get(c,0)+q\n            actions.append(action); continue\n        selected=None; selected_score=-1\n        if idx in assign:\n            target,op,value,c=tasks[assign[idx]]\n            if target not in reserved:\n                selected=(target,op,c)\n        if selected is None:\n            for ti,(target,op,value,c) in enumerate(tasks):\n                if target in reserved: continue\n                dist=distance(pos,target)\n                needed=1\n                if op==\'PLANT\':\n                    needed=2+(1 if tiles[target[1]][target[0]] is not None else 0)\n                    if seeds.get(c,0)<=0: continue\n                if op==\'FEED\' and inv.get(\'WHEAT\',0)<=0: continue\n                if op==\'PLACE\' and inv.get(c,0)<=0: continue\n                if dist+needed>remaining: continue\n                if day==final_day and op==\'WATER\':\n                    delivery=min(distance(target,p) for p in shed_positions)\n                    if dist+3+delivery>remaining:\n                        op=\'HARVEST\'\n                if day==final_day and op==\'HARVEST\':\n                    delivery=min(distance(target,p) for p in shed_positions)\n                    if dist+1+delivery+1>remaining: continue\n                # Autoregressive: crowd and op_count describe the assignments\n                # already made this turn, so later units see earlier decisions.\n                crowd=sum(1 for t in reserved if distance(t,target)<=2)\n                z=(base_dot[ti]+W[8]*(dist/10.0)+W[12]*min(1.0,crowd/4.0)\n                   +W[13]*min(1.0,op_count.get(op,0)/4.0))\n                if z>3.0: z=3.0\n                elif z<-3.0: z=-3.0\n                score=value/(dist+TUNE[\'travel_cost\'])*math.exp(z)\n                if score>selected_score: selected_score=score; selected=(target,op,c)\n        if selected:\n            target,op,c=selected; reserved.add(target)\n            op_count[op]=op_count.get(op,0)+1\n            if tuple(pos)!=target: action=move(pos,target)\n            elif op==\'PLANT\':\n                if tiles[target[1]][target[0]] is not None: action=[\'DIG\']\n                else:\n                    action=[\'PLANT\',c]; seeds[c]-=1\n            elif op==\'PICKUP\': action=[\'PICKUP\',c[0],int(c[1])]\n            elif op==\'PLACE\': action=[\'PLACE\',c]\n            else: action=[op]\n        elif carrying:\n            action=[\'DROP\'] if home_dist==0 else move(pos,closest)\n            if home_dist==0:\n                for c,q in inv.items(): expected_drop[c]=expected_drop.get(c,0)+q\n        else: action=[\'PASS\']\n        actions.append(action)\n    # Count animals already in transit, not just those sitting in the shed:\n    # a pen a unit is walking toward is not a pen that needs restocking.\n    pending=(sum(shed.get(a,0) for a in ANIMALS)\n             + sum(i.get(a,0) for i in inventories for a in ANIMALS))\n    if day<=TUNE[\'animal_last_day\'] and n_pens>pending:\n        # Milk and wool are separate thin markets, so a herd of one species\n        # saturates its own price while the other sits unsupplied. sheep_share\n        # sets how much of the herd is sheep; 0 keeps the cow-only behaviour.\n        total=_tot\n        # Buying a goose when only pastures stand leaves it in the shed for\n        # ever: the crew picks it up, cannot place it, and end-of-day auto-drop\n        # returns it. Only buy what there is somewhere to put.\n        a=want_animal\n        if empty_pens.get(ANIMALS[a][\'pen\'],0)<=0:\n            a=None\n            for _cand in ANIMALS:\n                if empty_pens.get(ANIMALS[_cand][\'pen\'],0)>0: a=_cand; break\n        # Reserve the feed this herd will actually consume rather than a flat\n        # constant. Each animal eats 2 wheat a day, so an animal bought on day d\n        # commits ~2*price*(days left) on top of its purchase price. A fixed\n        # threshold ignores that and is a knife edge: 2303 measured 87k while\n        # 2400 measured 21k, because affording the Nth cow on one exact turn\n        # cascades through the whole season.\n        if a is None: a=\'COW\'; n_pens=pending\n        wprice=max(1,prices.get(\'WHEAT\',25))\n        horizon=max(0,min(final_day-day,TUNE[\'feed_horizon\']))\n        reserve=(total+1)*2*wprice*horizon*TUNE[\'feed_safety\']\n        afford=int(max(0,(money-reserve)//ANIMALS[a][\'cost\']))\n        qty=min(n_pens-pending,afford,3)\n        if qty>0:\n            orders.append([\'BUY_ANIMAL\',a,qty]); money-=qty*ANIMALS[a][\'cost\']\n    if n_animals>0 and day!=final_day:\n        wp=max(1,prices.get(\'WHEAT\',25))\n        # Buy a buffer, not one day\'s ration. An animal unfed twice escapes,\n        # taking its cost and every future yield; in real games the herd was\n        # lost outright in 3 of 5 episodes while the winners bought 235 wheat\n        # to our 31. One thin turn -- a dip in cash, or a market order pushed\n        # off the end of the cap -- is enough to start that.\n        need=int(n_animals*2*TUNE[\'feed_buffer\']-shed.get(\'WHEAT\',0)-carried_wheat)\n        qty=max(0,min(need,int(max(0,(money-200)//wp)),int(TUNE[\'feed_max\'])))\n        if qty>0:\n            keep.append([\'BUY_PRODUCT\',\'WHEAT\',qty]); money-=qty*wp\n\n    for c in PARAMS:\n        quantity=shed.get(c,0)+expected_drop.get(c,0)\n        if c==\'WHEAT\' and n_animals>0 and day!=final_day:\n            # Wheat held back as feed; selling it just to re-buy it next turn\n            # churns market-order slots for nothing.\n            # Hold back the SAME buffer the feed purchase targets. Reserving\n            # one day while buying two made the farm buy 1,718 wheat and sell\n            # 1,625 of it straight back in a real game: a loss on every unit\n            # and a market-order slot burned every turn.\n            quantity-=int(n_animals*2*TUNE[\'feed_buffer\'])\n        if quantity<=0: continue\n        inv_now=market_inv.get(c,10000); base=PARAMS[c][0]\n        if days_left<=TUNE[\'sell_clear\']:\n            qty=quantity\n        else:\n            # Units sellable before the marginal price falls through the floor.\n            cap=0\n            while cap<quantity and price(c,inv_now+cap,overrides)>=TUNE[\'sell_floor\']*base:\n                cap+=1\n            # Never stall: stock still has to clear before the season ends.\n            must=-(-quantity//max(1,int(min(days_left,TUNE[\'sell_spread\']))))\n            qty=min(quantity,max(cap,must))\n        if qty>0: orders.append([\'SELL\',c,int(qty)])\n    # Reserve cash for seeds, workers and ongoing operation before expansion.\n    if hour<2 and unlocked<TUNE[\'max_land\'] and day<TUNE[\'land_last_day\'] and active>=unlocked*(n//2)**2*TUNE[\'land_fill\']:\n        cost=[1000,2000,4000][unlocked-1]\n        if money>cost+TUNE[\'land_cash\']:\n            keep.append([\'BUY_LAND\']); money-=cost\n    workers=TUNE[\'workers_hi\'] if active>TUNE[\'active_hi\'] else (TUNE[\'workers_mid\'] if active>TUNE[\'active_lo\'] else TUNE[\'workers_lo\'])\n    if day==0: workers=TUNE[\'workers_mid\']\n    if best_value<0 and active==0: workers=0\n    if hour<=2 and remaining>10:\n        fib=[1,1,2,3,5,8,13,21,34,55,89,144]\n        for hire in range(farm.get(\'hires_today\',0),int(workers)):\n            cost=fib[hire]*cfg.get(\'farmHandCostMult\',1)\n            # Feed is a standing daily bill and an animal unfed for two days\n            # escapes, taking 400 coins and every future tick of milk with it.\n            # A marginal hand is worth a few coins a day, so payroll yields to\n            # the herd: at 10 hands the farm ran out of wheat money by day 8 and\n            # had lost the whole herd by day 22.\n            if money-cost < feed_hold+50: break\n            keep.append([\'HIRE\']); money-=cost\n    # Buffer only a few seeds, avoiding capital tied up in obsolete crop choices.\n    if best_value>2 and hour<tpd-3:\n        free_tiles=sum(t is None or isinstance(t,dict) and t.get(\'kind\')==\'WEED\' for row in tiles for t in row)\n        free_tiles-=sum(a[0]==\'PLANT\' for a in actions)\n        desired=min(max(0,free_tiles),int(TUNE[\'seed_cap\']),max(2,len(positions)))\n        needed=max(0,desired-seeds.get(best_crop,0))\n        affordable=max(0,int((money-60)//CROPS[best_crop][0]))\n        quantity=min(needed,affordable)\n        if quantity: keep.append([\'BUY_SEED\',best_crop,quantity])\n    # Orders past maxMarketOrdersPerTurn are silently discarded. Sells go first\n    # so their cash is available to the buys behind them in the same turn, but\n    # they must never crowd out payroll or investment: a morning with a full\n    # shed produces many sell orders, and at nine hands that pushed BUY_SEED off\n    # the end entirely -- the farm stopped sowing and scores halved.\n    cap=int(cfg.get(\'maxMarketOrdersPerTurn\',10))\n    # Within the protected list, hires yield last: a dropped HIRE costs one\n    # hand for one day, a dropped BUY_SEED costs a tile for the rest of the\n    # season. Stable sort, so the rest keep their order.\n    keep.sort(key=lambda o: 1 if o[0]==\'HIRE\' else 0)\n    room=max(0,cap-len(keep))\n    orders.sort(key=lambda o:-(prices.get(o[1],0)*o[2]) if o[0]==\'SELL\' else 0)\n    market=(orders[:room]+keep)[:cap]\n    return {\'farmer\':actions[0],\'hands\':actions[1:],\'market\':market}\n'
Path("main.py").write_text(AGENT_SOURCE)
print("Created main.py")

In [ ]:
from kaggle_environments import make

env = make("kaggriculture", configuration={"seed": 601}, debug=True)
env.run(["main.py", "main.py"])
statuses = [s.status for s in env.state]
assert statuses == ["DONE", "DONE"], statuses
print("Self-play validation:", [(s.status, s.reward) for s in env.state])

In [ ]:
import tarfile

with tarfile.open("submission.tar.gz", "w:gz") as archive:
    archive.add("main.py", arcname="main.py")
print("Ready: submission.tar.gz")